# BHT Agentic Pipeline Notebook (Anaconda + Azure OpenAI)

This notebook shows how to read an SPSS `.sav` file and run `bht_agentic_pipeline.py` using Azure OpenAI settings from an environment file (`.env` or `.evn`).

In [ ]:
# If needed once in this kernel
# !pip install pyreadstat pandas requests python-dotenv




In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_file = Path('.env')
if not env_file.exists() and Path('.evn').exists():
    env_file = Path('.evn')  # supports your requested filename typo as well

load_dotenv(env_file if env_file.exists() else None)
print('Loaded env file:', env_file if env_file.exists() else 'None found')




In [ ]:
# Required Azure env vars
required = [
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_DEPLOYMENT',
    'AZURE_OPENAI_API_VERSION',
]
for k in required:
    print(k, 'OK' if os.getenv(k) else 'MISSING')




In [ ]:
import pyreadstat
sav_path = './data/your_file.sav'  # <-- update this
df, meta = pyreadstat.read_sav(sav_path, apply_value_formats=False)
print('Rows:', len(df), 'Cols:', len(df.columns))
df.head(3)




In [ ]:
import subprocess
cmd = [
    'python', 'bht_agentic_pipeline.py', sav_path,
    '--outdir', 'outputs_notebook',
    '--provider', 'azure'
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)




In [ ]:
import json
from pathlib import Path
out = Path('outputs_notebook')
for name in ['master_metadata.json', 'column_mapping.json', 'questionnaire_logic.json']:
    p = out / name
    print(name, 'exists:', p.exists())

sample = json.loads((out / 'master_metadata.json').read_text())
list(sample.items())[:5]




## Step 0: Missing Value Intelligence Agent
Run the Step 0 agent and export 3 CSV outputs under `data/outputs`.


In [ ]:
from agent.step0_missing_value_intelligence import MissingValueIntelligenceAgent, save_step0_outputs
from pathlib import Path
import json

out_dir = Path('outputs_notebook')
column_mapping_path = out_dir / 'column_mapping.json'
questionnaire_logic_path = out_dir / 'questionnaire_logic.json'
column_mapping = json.loads(column_mapping_path.read_text()) if column_mapping_path.exists() else {}

agent = MissingValueIntelligenceAgent(
    questionnaire_logic_path=str(questionnaire_logic_path) if questionnaire_logic_path.exists() else None,
)

valid_nulls, invalid_missing, uncertain_missing = agent.run(df, column_mapping)
paths = save_step0_outputs(
    valid_nulls=valid_nulls,
    invalid_missing=invalid_missing,
    uncertain_missing=uncertain_missing,
    outdir='data/outputs',
)

print('Step 0 Excel outputs:')
for k, v in paths.items():
    print(f'- {k}: {v}')
print('counts =>', len(valid_nulls), len(invalid_missing), len(uncertain_missing))






## Step 1: Row-Level Anomaly Detection
Detect row-level issues using `master_metadata.json` + `column_mapping.json` and export `anomaly.csv`.


In [ ]:
from agent.step1_row_level_anomaly_detection import DetectionAgent, save_step1_output
import json
from pathlib import Path

out_dir = Path('outputs_notebook')
metadata_path = out_dir / 'master_metadata.json'
column_mapping_path = out_dir / 'column_mapping.json'

if not metadata_path.exists() or not column_mapping_path.exists():
    raise FileNotFoundError('Run the main pipeline cell first to create metadata + mapping JSON files.')

metadata = json.loads(metadata_path.read_text())
column_mapping = json.loads(column_mapping_path.read_text())

detector = DetectionAgent()
anomalies = detector.run(df=df, metadata=metadata, column_mapping=column_mapping)
anomalies_df = pd.DataFrame(anomalies)
print('Step 1 anomaly count:', len(anomalies))
anomaly_path = save_step1_output(anomalies, outdir='data/outputs')
print('Saved:', anomaly_path)
anomalies_df.head()






## Step 2: Column Health Classification
Classify columns as `SYSTEMIC` vs `ROW_LEVEL` from Step 1 anomalies and save `column_health_report.csv`.


In [ ]:
from agent.step2_column_health_classification import ColumnHealthAgent, save_step2_output

column_health_agent = ColumnHealthAgent(threshold=0.25)
column_health = column_health_agent.run(
    anomalies=anomalies,
    total_rows=df.shape[0],
)
report_path = save_step2_output(column_health, outdir='data/outputs')

print('Step 2 rows:', len(column_health))
print('Saved:', report_path)
column_health.head()





## Step 3: Decision Routing
Route anomalies to LLM queue vs human review queue using Step 2 column health status.


In [ ]:
from agent.step3_decision_routing import DecisionRoutingAgent

routing_agent = DecisionRoutingAgent(systemic_threshold=0.25)
llm_queue, human_queue = routing_agent.run(
    anomalies=anomalies,
    column_health=column_health
)

print('LLM anomalies:', len(llm_queue))
print('Human review anomalies:', len(human_queue))





## Step 4: Rule Engine (Deterministic Auto-Fix)
Apply deterministic safe fixes and export auto-decisions for governance.


In [ ]:
from agent.step4_rule_engine import RuleEngine
from pathlib import Path

rule_engine = RuleEngine()
column_health_map = dict(zip(column_health['column'], column_health['status']))

auto_decisions = []
remaining_for_llm = []

for a in llm_queue:
    rule_decision = rule_engine.auto_resolve(a, column_health_map)

    if rule_decision:
        auto_decisions.append({
            'anomaly': a,
            'action': {
                'type': rule_decision['recommended_action'],
                'parameters': {'method': 'median'}
            },
            'confidence': rule_decision['confidence'],
            'reasoning': 'Deterministic rule'
        })
    else:
        remaining_for_llm.append(a)

print('Auto-resolved:', len(auto_decisions))
print('To LLM:', len(remaining_for_llm))

auto_decisions_df = pd.DataFrame([
    {
        'row_index': d['anomaly'].get('row_index'),
        'column': d['anomaly'].get('column'),
        'metric': d['anomaly'].get('metric'),
        'issue': d['anomaly'].get('issue_type'),
        'action': d['action']['type'],
        'confidence': d['confidence'],
        'source': 'RULE_ENGINE'
    }
    for d in auto_decisions
])

out_dir = Path('data/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
auto_path = out_dir / 'auto_decisions.csv'
auto_decisions_df.to_csv(auto_path, index=False)
print('Auto decisions exported:', auto_path)





## Step 5A: LLM Decisions (Clustered, Policy-Level)
Run one LLM reasoning call per anomaly pattern cluster and export `llm_decisions.csv`.


In [ ]:
from agent.step5a_llm_policy_decision import DecisionAgentV2
from pathlib import Path
import json

try:
    from openai import AzureOpenAI
except Exception as e:
    raise ImportError('Install openai package: pip install openai') from e

out_dir = Path('outputs_notebook')
variable_labels = json.loads((out_dir / 'variable_labels.json').read_text())
metadata = json.loads((out_dir / 'master_metadata.json').read_text())

azure_client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-06-01'),
)
deployment_name = os.getenv('AZURE_OPENAI_DEPLOYMENT')
if not deployment_name:
    raise ValueError('Missing AZURE_OPENAI_DEPLOYMENT in environment.')

decision_agent = DecisionAgentV2(
    llm_client=azure_client,
    deployment_name=deployment_name,
    auto_exec_threshold=0.85
)

llm_decisions = decision_agent.run(
    anomalies=remaining_for_llm,
    variable_labels=variable_labels,
    metadata=metadata
)

print('LLM decisions:', len(llm_decisions))

llm_decisions_df = pd.DataFrame([
    {
        'row_index': d['anomaly'].get('row_index'),
        'column': d['anomaly'].get('column'),
        'metric': d['anomaly'].get('metric'),
        'issue': d['anomaly'].get('issue_type'),
        'action': d['action'].get('type'),
        'confidence': d.get('confidence'),
        'reasoning': d.get('reasoning'),
        'source': 'LLM_POLICY_AGENT'
    }
    for d in llm_decisions
])

out_excel = Path('data/outputs/llm_decisions.csv')
out_excel.parent.mkdir(parents=True, exist_ok=True)
llm_decisions_df.to_csv(out_excel, index=False)
print('LLM decisions exported:', out_excel)





## Step 5B: Missing Value LLM (Only Uncertain Cases)


In [ ]:
from agent.step5b_missing_value_llm import MissingValueLLMAgentV2
from pathlib import Path
import json

client = globals().get('client', globals().get('azure_client'))
if client is None:
    raise ValueError('No LLM client found. Expected `azure_client` (or `client`) from Step 5A setup.')

out_dir = Path('outputs_notebook')
variable_labels = json.loads((out_dir / 'variable_labels.json').read_text())
column_mapping = json.loads((out_dir / 'column_mapping.json').read_text())

missing_llm_agent = MissingValueLLMAgentV2(
    client=client,
    deployment_name=deployment_name,
    max_llm_columns=10  # adjust as needed
)

missing_llm_decisions = missing_llm_agent.run(
    uncertain_cases=[
        {'row_index': r, 'column': c}
        for r, c in uncertain_missing
    ],
    variable_labels=variable_labels,
    column_mapping=column_mapping
)

print('Missing value LLM decisions:', len(missing_llm_decisions))

out_excel = Path('data/outputs/missing_llm_decisions.csv')
out_excel.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(missing_llm_decisions).to_csv(out_excel, index=False)
print('Missing-value decisions exported:', out_excel)





## Step 6: Governed Execution (Single Control Point)


In [ ]:
from agent.step6_execution_agent import ExecutionAgentV2
from pathlib import Path

execution_agent = ExecutionAgentV2(auto_exec_threshold=0.85)
all_decisions = auto_decisions + llm_decisions

cleaned_df, audit_log = execution_agent.run(
    df=df,
    decisions=all_decisions,
    missing_value_decisions=missing_llm_decisions
)

display(audit_log.head())

out_dir = Path('data/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(out_dir / 'cleaned_df.csv', index=False)
audit_log.to_csv(out_dir / 'audit_log.csv', index=False)

print('Pipeline completed successfully.')






## Step 7A: Resolve Baseline Stats


## Step 7B: Compute Current Stats


## Step 7C: Drift Detection


## Step 8: Systemic Intelligence


In [ ]:
from agent.step7_stats_agent import StatsAgent
from agent.step7_baseline_stats_agent import BaselineStatsAgent
from agent.step7_drift_agent import DriftAgent1
from agent.step8_systemic_intelligence_agent import SystemicIntelligenceAgent
from pathlib import Path
import json

client = globals().get('client', globals().get('azure_client'))
if client is None:
    raise ValueError('No LLM client found. Expected `azure_client` (or `client`) from Step 5A setup.')

stats_agent = StatsAgent(bins=10)
baseline_agent = BaselineStatsAgent(stats_agent=stats_agent)
drift_agent = DriftAgent1(mean_z_thresh=3.0, std_pct_thresh=0.5, psi_thresh=0.25)
systemic_agent = SystemicIntelligenceAgent(llm_client=client, deployment_name=deployment_name)

out_dir = Path('data/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

out_notebook = Path('outputs_notebook')
metadata = json.loads((out_notebook / 'master_metadata.json').read_text())
column_mapping = json.loads((out_notebook / 'column_mapping.json').read_text())

baseline_stats = baseline_agent.resolve(
    metadata=metadata,
    column_mapping=column_mapping,
    baseline_data_path='data/historical/bht_baseline.csv',
)
print(f'Baseline stats resolved for {len(baseline_stats)} columns.')

current_stats = stats_agent.compute_stats(
    df=df,
    metadata=metadata,
    column_mapping=column_mapping,
)
print(f'Computed current stats for {len(current_stats)} columns.')

drift_alerts = drift_agent.detect(
    current_stats=current_stats,
    historical_stats=baseline_stats,
)
print('Drift alerts detected:', len(drift_alerts))

pd.DataFrame(drift_alerts).to_csv(out_dir / 'drift_alerts.csv', index=False)
with open(out_dir / 'current_stats.json', 'w', encoding='utf-8') as f:
    json.dump(current_stats, f, indent=2)
print('Drift outputs saved.')

executive_summary = systemic_agent.generate_recommendations(
    column_health_report=column_health,
    drift_alerts=drift_alerts,
)
with open(out_dir / 'systemic_executive_summary.txt', 'w', encoding='utf-8') as f:
    f.write(executive_summary)

print(executive_summary)





